# V4 — Gaussian blur shared-memory tile + halo

Owner: Thịnh. V4 kế thừa **đúng V3 đã merge**, chỉ thay Gaussian blur. Nền V3 đã tích hợp V2 S4 và V3 S6; V4 giữ nguyên các stage này.

Chọn Colab **GPU**, rồi Run all sau khi source V4 đã được push lên branch `v4`. Notebook không chứa số đo giả định. Chốt hiệu quả V4 chỉ sau parity, 12 case timing và profiler V3/V4.


In [ ]:
from pathlib import Path
import csv
import datetime
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from statistics import geometric_mean

REPO_URL = "https://github.com/nganbee/panorama-stitching-project.git"
BRANCH = "v4"
REPO = Path("/content/panorama-stitching-project")
UPDATE_REPO = True  # False: giữ source đã upload/chỉnh tại Colab.
RUNS = 5
RUN_PROFILER = True  # Có nsys thì profile; thiếu tool ghi NOT RUN.
RUN_REVERSE_SWEEP = False  # True: đo thêm một sweep đảo thứ tự V3/V4.
COMMAND_LOG_DIR = None
COMMAND_INDEX = 0

def run(args, check=True, log=None):
    # Capture + print explicitly so subprocess output is retained in the ipynb.
    global COMMAND_INDEX
    command = shlex.join([str(a) for a in args])
    print("$", command, flush=True)
    if log is None and COMMAND_LOG_DIR is not None:
        COMMAND_INDEX += 1
        log = COMMAND_LOG_DIR / f"command_{COMMAND_INDEX:03d}.log"
    stream = None
    try:
        if log is not None:
            log = Path(log)
            log.parent.mkdir(parents=True, exist_ok=True)
            stream = log.open("w", encoding="utf-8")
            stream.write("$ " + command + "\n")
        with subprocess.Popen([str(a) for a in args], text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                if stream is not None:
                    stream.write(line)
                    stream.flush()
            code = process.wait()
    finally:
        if stream is not None:
            stream.close()
    if check and code:
        raise RuntimeError(f"Command failed ({code}); log: {log}")
    return code

if not REPO.exists():
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
if UPDATE_REPO:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
    if dirty:
        print("Có thay đổi local: giữ nguyên checkout, không tự pull.")
    else:
        run(["git", "fetch", "origin", BRANCH])
        run(["git", "checkout", BRANCH])
        ancestor = subprocess.run(["git", "merge-base", "--is-ancestor", "HEAD", f"origin/{BRANCH}"])
        if ancestor.returncode:
            raise RuntimeError("Checkout đã lệch lịch sử (có thể sau rebase). Dùng thư mục REPO mới để clone; không tự reset mất dữ liệu.")
        run(["git", "merge", "--ff-only", f"origin/{BRANCH}"])
required = ["src/cuda/v2_pipeline.cu", "src/stage_implementations/v2_stage4.cu",
            "src/cuda/v3_pipeline.cu", "src/stage_implementations/v4_stage1.cu"]
assert all(Path(p).is_file() for p in required), "Thiếu source V2/V3/V4 tích lũy."
assert RUNS >= 5
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
SESSION_DIR = Path("out") / f"v4_inspect_{STAMP}"
SESSION_DIR.mkdir(parents=True)
COMMAND_LOG_DIR = SESSION_DIR / "commands"
RESULTS = SESSION_DIR / "benchmark"  # benchmark script sẽ tạo thư mục này.
Path("build").mkdir(exist_ok=True)
run(["git", "status", "--short"])
run(["git", "branch", "--show-current"])
run(["git", "rev-parse", "HEAD"])
print("Session:", SESSION_DIR)


## 1. GPU, compiler và build chung

Giữ `--fmad=false`, không dùng fast math. Một binary cung cấp V0/V1/V2/V3/V4; mọi bản dùng cùng input, flags và GPU session. Tile mặc định 16×16. Có thể build lại với tile 32×8 hoặc 32×16 qua `TILE_FLAGS`, chạy lại toàn bộ parity và lưu kết quả vào thư mục riêng.


In [ ]:
run(["nvidia-smi"], log=SESSION_DIR / "nvidia-smi.log")
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name,uuid,driver_version",
                               "--format=csv,noheader"], text=True).strip()
print("GPU:", GPU)
run(["nvcc", "--version"])
run(["g++", "--version"])
TILE_FLAGS = []  # e.g. ["-DPANORAMA_V4_TILE_X=32", "-DPANORAMA_V4_TILE_Y=8"]
FLAGS = ["nvcc", "-O2", "-std=c++17", "--fmad=false",
         "-DPANORAMA_WITH_CUDA", "-DPANORAMA_WITH_V2", "-DPANORAMA_WITH_V3", "-DPANORAMA_WITH_V4",
         "-Isrc/core_headers", "-Isrc/libraries"] + TILE_FLAGS
SOURCES = [
    "src/integration/v0_pipeline.cpp",
    "src/stage_implementations/v0_stage12.cpp",
    "src/stage_implementations/v0_stage34.cpp",
    "src/stage_implementations/v0_stage5.cpp",
    "src/stage_implementations/v0_stage6.cpp",
    "src/stage_implementations/v1_stage12.cu",
    "src/stage_implementations/v1_stage34.cu",
    "src/stage_implementations/v1_stage5.cu",
    "src/stage_implementations/v1_stage6.cu",
    "src/stage_implementations/v2_stage4.cu",
    "src/stage_implementations/v3_stage6.cu",
    "src/stage_implementations/v4_stage1.cu",
    "src/cuda/cuda_pipeline.cu", "src/cuda/v2_pipeline.cu", "src/cuda/v3_pipeline.cu", "src/cuda/v4_pipeline.cu",
    "src/libraries/stb_impl.cpp",
]
run(FLAGS + ["src/integration/main.cpp"] + SOURCES + ["-o", "build/stitch_v4"])
run(["build/stitch_v4", "--list-versions"])
run(["build/stitch_v4", "--dump-params"])
(SESSION_DIR / "build_flags.json").write_text(json.dumps(FLAGS, indent=2), encoding="utf-8")
(SESSION_DIR / "environment.json").write_text(json.dumps({
    "gpu": GPU, "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "worktree": subprocess.check_output(["git", "status", "--short"], text=True),
    "flags": FLAGS, "runs": RUNS, "warmup_runs": 1,
}, indent=2), encoding="utf-8")


## 2. Parity blur riêng và lan truyền S2–S4

Test V0/V3/V4, ảnh hằng, impulse góc, checkerboard, random, ảnh 1 pixel/nhỏ hơn halo, kích thước không chia hết tile và input lỗi. Yêu cầu V3/V4 blur bit-identical với cùng flags; so V0 theo `atol=2e-4, rtol=2e-6`. Kiểm tra tiếp keypoints, descriptor bits và ordered matches.


In [ ]:
run(FLAGS + [
    "src/tools/v4/test_v4_blur.cpp",
    "src/stage_implementations/v0_stage12.cpp", "src/stage_implementations/v0_stage34.cpp",
    "src/stage_implementations/v1_stage12.cu", "src/stage_implementations/v1_stage34.cu",
    "src/stage_implementations/v2_stage4.cu",
    "src/stage_implementations/v4_stage1.cu",
    "-o", "build/test_v4_blur",
])
run(["build/test_v4_blur"])
run(["build/stitch_v4", "--selftest", "--version", "4"])


## 3. Parity toàn pipeline V4 so với V0/V3

Test identity, dịch chuyển, input lỗi và no-feature; sau đó chạy lại để kiểm tra trạng thái profiler. So keypoints/matches/inliers, sai số chiếu H ≤1e-3 px, kích thước crop, RGB ≤1 mức so với V0 và RGB bit-identical V3/V4. Test timing kiểm tra tổng stage bằng kernel_total. Case V0 thất bại không được đánh dấu panorama parity PASS.


In [ ]:
run(FLAGS + ["src/tools/v4/test_v4_pipeline.cpp"] + SOURCES + ["-o", "build/test_v4_pipeline"])
run(["build/test_v4_pipeline"])


## 4. Compute Sanitizer

Chạy cả memcheck và synccheck cho test blur để phát hiện truy cập halo và lỗi barrier; pipeline test chạy memcheck. Nếu tool không có, ghi rõ NOT RUN thay vì PASS.


In [ ]:
if shutil.which("compute-sanitizer"):
    run(["compute-sanitizer", "--tool", "memcheck", "--error-exitcode", "1", "build/test_v4_blur"])
    run(["compute-sanitizer", "--tool", "synccheck", "--error-exitcode", "1", "build/test_v4_blur"])
    run(["compute-sanitizer", "--tool", "memcheck", "--error-exitcode", "1", "build/test_v4_pipeline"])
else:
    print("NOT RUN: Compute Sanitizer unavailable. Chưa nghiệm thu memory/synchronization safety.")


## 5. Baseline V3: xác nhận tỷ trọng blur trước phép đo V4

CLI tự chạy **1 warm-up + tối thiểu 5 lần đo**, JSON là trung bình. `blur` gồm sinh coefficients + horizontal + vertical cho cả A/B. Đây là timing kernel; H2D/D2H riêng, E2E gồm allocation/launch/sync/copy. Đọc/ghi ảnh nằm ngoài E2E. Cần profiler ở mục 8 để xác nhận blur thực sự đáng tối ưu.


In [ ]:
BASELINE = SESSION_DIR / "v3_baseline.json"
LEFT = "dataset/location1/large_4k/1.jpg"
RIGHT = "dataset/location1/large_4k/2.jpg"
run(["build/stitch_v4", LEFT, RIGHT, SESSION_DIR / "v3_baseline.png",
     "--version", "3", "--runs", RUNS, "--json", BASELINE], log=SESSION_DIR / "v3_baseline.log")
baseline = json.loads(BASELINE.read_text(encoding="utf-8"))
t = baseline["timings_ms"]
print("GPU:", GPU)
print(f"V3 blur={t['blur']:.4f} ms; E2E={t['end_to_end']:.4f} ms")
print(f"Blur share: {100*t['blur']/t['kernel_total']:.2f}% of kernel; "
      f"{100*t['blur']/t['end_to_end']:.2f}% of E2E")
for name, ms in sorted(baseline["stage_timings_ms"].items(), key=lambda p: p[1], reverse=True):
    print(f"{name}: {ms:.4f} ms ({100*ms/t['kernel_total']:.2f}% kernel)")
print("Bottleneck phải suy ra từ bảng, không mặc định là blur. Bảng đủ 12 case ở mục 7.")


## 6. Đủ 12 case: correctness + V0/V3/V4 timing + JSON

Mỗi case chạy parity riêng, rồi V0/V3/V4 có warm-up và ≥5 timed runs. Mọi ca lỗi được lưu log, ghi failed; script vẫn đo các case còn lại. `summary.csv/json` chứa stage bottleneck, blur/E2E speedup và transfer timings. Kết quả tăng tốc dưới 1 được giữ để phân tích. Tránh so số đo lấy từ các GPU session khác nhau.


In [ ]:
benchmark_status = run([sys.executable, "src/tools/v4/benchmark_v4.py",
    "--binary", "build/stitch_v4", "--parity", "build/test_v4_pipeline",
    "--runs", RUNS, "--output", RESULTS], check=False, log=SESSION_DIR / "benchmark.log")
print("Benchmark exit code:", benchmark_status)
if not (RESULTS / "summary.csv").is_file():
    raise RuntimeError("Benchmark chưa tạo summary.csv; xem benchmark.log.")
if RUN_REVERSE_SWEEP:
    REVERSE_RESULTS = SESSION_DIR / "benchmark_reverse"
    reverse_status = run([sys.executable, "src/tools/v4/benchmark_v4.py",
        "--binary", "build/stitch_v4", "--parity", "build/test_v4_pipeline",
        "--runs", RUNS, "--reverse-order", "--output", REVERSE_RESULTS],
        check=False, log=SESSION_DIR / "benchmark_reverse.log")
    print("Reverse sweep exit code:", reverse_status)


## 7. Inspect timing V0/V3/V4 — từng bước, stage và E2E

Mọi bảng dùng JSON **cùng session**, không ghép số đo từ notebook V3 cũ. Đổi RESULTS thành thư mục kết quả cũ rồi chạy lại mục này để inspect mà không cần benchmark lại. CSV đầy đủ xuất vào RESULTS/inspection/.

S1 = gray + blur; S2 = harris + nms_topk. Thời gian GPU trong JSON là CUDA-event; V0 dùng CPU timer. Kernel total đã bao gồm các bước GPU, không cộng lại. Residual = E2E − kernel − H2D − D2H, gồm allocation/free, điều phối, synchronization và phần chưa hạch toán; không gọi residual là kernel hay thời gian CPU thuần.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)
TIMING_FIELDS = ["gray", "blur", "harris", "nms_topk", "describe", "match", "ransac",
                 "warp_blend", "kernel_total", "h2d", "d2h", "end_to_end"]
STEPS = TIMING_FIELDS[:8]
STAGES = ["S1", "S2", "S3", "S4", "S5", "S6"]
INSPECTION_DIR = RESULTS / "inspection"
INSPECTION_DIR.mkdir(exist_ok=True)
summary = pd.read_csv(RESULTS / "summary.csv")
environment_path = RESULTS / "environment.json"
environment = json.loads(environment_path.read_text(encoding="utf-8")) if environment_path.is_file() else {}
GPU_LABEL = environment.get("gpu_identity", environment.get("gpu", "Không có GPU metadata trong thư mục này"))
print("Reports:", RESULTS, "\nGPU:", GPU_LABEL)
display(summary[["case", "status", "parity"]])
failures = summary.loc[summary.status.ne("ok") | summary.parity.ne("pass")]
print(f"Verified successful cases: {len(summary)-len(failures)}/{len(summary)}; expected: 12")
if not failures.empty:
    display(failures)
    print("Giữ ca lỗi trong báo cáo; không tính speedup hoặc tuyên bố parity PASS cho ca đó.")
records = []
for item in summary.to_dict("records"):
    for version in (0, 3, 4):
        report_path = RESULTS / f"{item['case']}_v{version}.json"
        if not report_path.is_file():
            continue
        try:
            report = json.loads(report_path.read_text(encoding="utf-8"))
        except (OSError, ValueError) as error:
            report = {"status": "invalid_json", "error": str(error)}
        record = {"case": item["case"], "resolution": item["case"].rsplit("_", 1)[-1],
                  "version": version, "status": report.get("status", "unknown"),
                  "parity": item["parity"], "case_status": item["status"],
                  "runs": report.get("runs"), "warmup_runs": report.get("warmup_runs"),
                  "report": str(report_path)}
        record.update(report.get("timings_ms", {}))
        record.update({"stage_"+name: ms for name, ms in report.get("stage_timings_ms", {}).items()})
        record.update(report.get("features", {}))
        records.append(record)
if not records:
    raise RuntimeError("Thiếu JSON timing gốc. summary.csv cũ không đủ khôi phục mọi bước, đặc biệt Harris/NMS.")
timings = pd.DataFrame(records).reindex(columns=list(dict.fromkeys(
    ["case", "resolution", "version", "status", "parity", "case_status", "runs", "warmup_runs"] +
    TIMING_FIELDS + ["stage_"+s for s in STAGES] + [k for r in records for k in r])))
timings["residual_ms"] = float("nan")
gpu_rows = timings.version.isin([3, 4]) & timings.status.eq("ok")
timings.loc[gpu_rows, "residual_ms"] = (timings.end_to_end - timings.kernel_total - timings.h2d - timings.d2h)
timings.to_csv(INSPECTION_DIR / "timings_detailed.csv", index=False)
for version in (0, 3, 4):
    print(f"V{version}: tất cả bước, đơn vị ms; trung bình >=5 runs")
    display(timings.loc[timings.version.eq(version), ["case", "status", "parity"] + TIMING_FIELDS].round(4))
for version in (0, 3, 4):
    print(f"V{version}: stages S1-S6 (ms)")
    display(timings.loc[timings.version.eq(version), ["case"] + ["stage_"+s for s in STAGES]].round(4))
print("V3/V4: kernel, copy và residual — tổng bằng E2E theo định nghĩa")
display(timings.loc[gpu_rows, ["case", "version", "kernel_total", "h2d", "d2h", "residual_ms", "end_to_end"]].round(4))
if timings.loc[gpu_rows, "residual_ms"].lt(0).any():
    print("Có residual âm: kiểm tra cách hạch toán/timer; không tự clip về 0.")


### 7.1 So sánh blur, E2E và giới hạn lợi ích

Delta = V3 − V4: dương là V4 giảm thời gian. e2e_if_only_blur_changed_ms giả định chỉ đổi thời gian kernel blur và mọi phần khác cố định. Không quy toàn bộ E2E speedup cho blur. Bảng theo độ phân giải dùng geometric mean của speedup từng case.

remaining_e2e_delta_ms là phần chênh E2E còn lại sau khi trừ delta kernel blur; không tự gọi toàn bộ phần này là nhiễu.


In [ ]:
verified = timings.loc[timings.status.eq("ok") & timings.case_status.eq("ok") & timings.parity.eq("pass")]
comparison_rows = []
stage_comparison_rows = []
for case, group in verified.groupby("case", sort=False):
    by_v = {int(r["version"]): r for r in group.to_dict("records")}
    if not all(v in by_v for v in (0, 3, 4)):
        continue
    v0, v3, v4 = [by_v[v] for v in (0, 3, 4)]
    for stage in STAGES:
        a,b = v3["stage_"+stage], v4["stage_"+stage]
        stage_comparison_rows.append({"case": case, "stage": stage, "V3_ms": a, "V4_ms": b,
                                      "saved_ms": a-b, "speedup": a/b if b>0 else None})
    comparison_rows.append({"case": case, "resolution": v3["resolution"],
        "v3_blur_ms": v3["blur"], "v4_blur_ms": v4["blur"],
        "blur_delta_ms": v3["blur"]-v4["blur"], "blur_speedup": v3["blur"]/v4["blur"],
        "v3_e2e_ms": v3["end_to_end"], "v4_e2e_ms": v4["end_to_end"],
        "e2e_delta_ms": v3["end_to_end"]-v4["end_to_end"],
        "remaining_e2e_delta_ms": (v3["end_to_end"]-v4["end_to_end"])-(v3["blur"]-v4["blur"]),
        "e2e_speedup": v3["end_to_end"]/v4["end_to_end"],
        "e2e_speedup_v0_v4": v0["end_to_end"]/v4["end_to_end"],
        "v3_blur_share_kernel_pct": 100*v3["blur"]/v3["kernel_total"],
        "v3_blur_share_e2e_pct": 100*v3["blur"]/v3["end_to_end"],
        "e2e_if_only_blur_changed_ms": v3["end_to_end"]-v3["blur"]+v4["blur"],
        "v3_largest_stage": max(STAGES, key=lambda s: v3["stage_"+s]),
        "v4_largest_stage": max(STAGES, key=lambda s: v4["stage_"+s])})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(INSPECTION_DIR / "comparison.csv", index=False)
pd.DataFrame(stage_comparison_rows).to_csv(INSPECTION_DIR / "stage_comparison_all_cases.csv", index=False)
if not comparison.empty:
    display(comparison[["case", "v3_blur_ms", "v4_blur_ms", "blur_delta_ms", "blur_speedup",
                        "v3_blur_share_kernel_pct", "v3_blur_share_e2e_pct", "v3_largest_stage", "v4_largest_stage"]].round(4))
    display(comparison[["case", "v3_e2e_ms", "v4_e2e_ms", "e2e_delta_ms", "e2e_speedup",
                        "e2e_speedup_v0_v4", "e2e_if_only_blur_changed_ms", "remaining_e2e_delta_ms"]].round(4))
    speedup_cols = ["blur_speedup", "e2e_speedup", "e2e_speedup_v0_v4"]
    by_resolution = comparison.groupby("resolution", sort=False)[speedup_cols].agg(geometric_mean)
    by_resolution.to_csv(INSPECTION_DIR / "comparison_by_resolution.csv")
    display(by_resolution.round(4))
    print(f"Blur V4 nhanh hơn: {(comparison.blur_speedup>1).sum()}/{len(comparison)} case.")
    print(f"E2E V4 nhanh hơn: {(comparison.e2e_speedup>1).sum()}/{len(comparison)} case.")
    print("Các case blur V4 chậm hơn:")
    display(comparison.loc[comparison.blur_speedup.lt(1), ["case", "v3_blur_ms", "v4_blur_ms", "blur_speedup"]].round(4))
    for cols, ylabel in [(["v3_blur_ms", "v4_blur_ms"], "Blur kernels (CUDA events, ms)"),
                          (["v3_e2e_ms", "v4_e2e_ms"], "End-to-end (host clock, ms)")]:
        comparison.set_index("case")[cols].plot.bar(figsize=(14,4), ylabel=ylabel)
        plt.tight_layout()
        plt.show()
else:
    print("Chưa có case đủ điều kiện so sánh. Xem failures/logs; không kết luận speedup.")


### 7.2 Inspect một case: toàn bộ bước, S1–S6 và bottleneck

Đổi INSPECT_CASE thành tên bất kỳ trong danh sách. Xem cả bước không đổi: nếu RANSAC/warp/copy biến động lớn hơn delta blur thì E2E chưa cô lập được hiệu quả blur. Các dòng step, stage và total là các cách phân rã khác nhau, không cộng chồng.


In [ ]:
print("Cases:", list(timings.case.unique()))
INSPECT_CASE = "location1_4k"

def inspect_case(case):
    chosen = timings.loc[timings.case.eq(case) & timings.status.eq("ok")]
    if chosen.empty:
        print("Không có timing thành công cho", case)
        return
    def compare_metrics(metrics):
        table = chosen.set_index("version")[metrics].T.reindex(columns=[0,3,4])
        table.columns = ["V0_ms", "V3_ms", "V4_ms"]
        table["saved_V3_minus_V4_ms"] = table.V3_ms-table.V4_ms
        table["speedup_V3_over_V4"] = table.V3_ms / table.V4_ms.where(table.V4_ms.gt(0))
        return table
    print("GPU:", GPU_LABEL, "\nCase:", case)
    if not chosen.parity.eq("pass").all() or not chosen.case_status.eq("ok").all():
        print("UNVERIFIED: timing vẫn được hiển thị để chẩn đoán; không dùng để kết luận tăng tốc.")
    step_table = compare_metrics(TIMING_FIELDS)
    stage_table = compare_metrics(["stage_"+s for s in STAGES])
    display(step_table.round(5))
    display(stage_table.round(5))
    display(chosen.reindex(columns=["version", "status", "parity", "runs", "warmup_runs",
                                   "keypoints_A", "keypoints_B", "matches", "inliers", "inlier_ratio"]))
    step_table.to_csv(INSPECTION_DIR / f"{case}_step_comparison.csv")
    stage_table.to_csv(INSPECTION_DIR / f"{case}_stage_comparison.csv")
    ranks = []
    for r in chosen.loc[chosen.version.isin([3,4])].to_dict("records"):
        for name in STEPS:
            ranks.append({"version": r["version"], "step": name, "ms": r[name],
                          "kernel_share_pct": 100*r[name]/r["kernel_total"],
                          "e2e_share_pct": 100*r[name]/r["end_to_end"]})
    if not ranks:
        return
    ranked = pd.DataFrame(ranks).sort_values(["version", "ms"], ascending=[True,False])
    print("Xếp hạng các bước kernel:")
    display(ranked.round(4))
    ranked.to_csv(INSPECTION_DIR / f"{case}_bottleneck_ranking.csv", index=False)
    stage_table[["V3_ms", "V4_ms"]].plot.barh(figsize=(9,4), xlabel="Stage kernel time (ms)")
    plt.tight_layout()
    plt.show()
    system = chosen.loc[chosen.version.isin([3,4])].set_index("version")
    system[["kernel_total", "h2d", "d2h", "residual_ms"]].plot.bar(stacked=True, figsize=(8,4), ylabel="ms")
    plt.title("E2E accounting; residual is not a kernel")
    plt.tight_layout()
    plt.show()
    for version in (3,4):
        image_path = RESULTS / f"{case}_v{version}.png"
        if image_path.is_file():
            print(f"Panorama V{version}")
            with Image.open(image_path) as image:
                display(image.copy())
inspect_case(INSPECT_CASE)


### 7.3 Kiểm tra thứ tự đo / chạy lại

Bật RUN_REVERSE_SWEEP=True ở cell đầu để thu thêm sweep đảo thứ tự. Bảng so độ ổn định của blur và E2E giữa hai sweep. JSON chỉ có trung bình, không có độ lệch chuẩn/min từng iteration; không suy ra ý nghĩa thống kê từ một số trung bình.


In [ ]:
if RUN_REVERSE_SWEEP and (REVERSE_RESULTS / "summary.csv").is_file():
    reverse_summary = pd.read_csv(REVERSE_RESULTS / "summary.csv")
    valid_forward = summary.loc[summary.status.eq("ok") & summary.parity.eq("pass")]
    valid_reverse = reverse_summary.loc[reverse_summary.status.eq("ok") & reverse_summary.parity.eq("pass")]
    order_columns = ["case", "blur_speedup_v3_v4", "e2e_speedup_v3_v4"]
    order_check = valid_forward[order_columns].merge(valid_reverse[order_columns], on="case", suffixes=("_forward", "_reverse"))
    order_check.to_csv(INSPECTION_DIR / "order_check.csv", index=False)
    display(order_check.round(4))
else:
    print("NOT RUN: reverse sweep. Bật RUN_REVERSE_SWEEP để kiểm tra bias thứ tự/clock.")


## 8. Profiler V3/V4: coefficients, horizontal, vertical và các kernel còn lại

RUN_PROFILER bật mặc định; nếu thiếu nsys thì lưu NOT RUN. Profile cùng case ở 1080p và 4K, cả V3 lẫn V4. Tách sinh coefficients, horizontal pass, vertical pass; hiện mọi kernel theo thời gian và lưu bảng CSV. Mỗi pass chạy hai lần/pipeline (ảnh A/B). Trace có cả warm-up + RUNS; thời gian per_pipeline là tổng trace chia RUNS+1, không phải phép đo loại warm-up. **Dùng JSON không bật profiler để báo cáo speedup**, dùng trace để giải thích công việc và bottleneck.


In [ ]:
PROFILE_DIR = SESSION_DIR / "profiles"
PROFILE_DIR.mkdir(exist_ok=True)
PROFILE_CASES = [("location1", "medium_1080p", "location1_1080p"),
                 ("location1", "large_4k", "location1_4k")]

def read_kernel_summary(output):
    rows, columns = [], None
    for values in csv.reader(output.splitlines()):
        values = [v.strip().lstrip("\ufeff") for v in values]
        if {"Total Time (ns)", "Instances", "Name"} <= set(values):
            columns = values
        elif columns is not None and len(values) == len(columns):
            row = dict(zip(columns, values))
            if row["Instances"].replace(",", "").isdigit():
                rows.append({"name": row["Name"],
                             "total_ns": float(row["Total Time (ns)"].replace(",", "")),
                             "instances": int(row["Instances"].replace(",", ""))})
    if not rows:
        raise RuntimeError("Không đọc được kernel CSV. Xem log Nsight; không đánh dấu profiling PASS.")
    return rows

def blur_part(name):
    if "gaussian_weights_kernel" in name:
        return "coefficients"
    if "gaussian_blur_kernel" in name or "gaussian_blur_tiled_kernel" in name:
        if re.search(r"<\s*(?:true|1|\(bool\)\s*1)\s*>", name):
            return "horizontal"
        if re.search(r"<\s*(?:false|0|\(bool\)\s*0)\s*>", name):
            return "vertical"
        raise RuntimeError("Không nhận diện được hướng pass từ kernel: " + name)
    return "other"

def profile_blur(scene, resolution, case, version):
    prefix = PROFILE_DIR / f"nsys_v{version}_{case}"
    left = Path("dataset") / scene / resolution / "1.jpg"
    right = left.with_name("2.jpg")
    print("PROFILE GPU:", GPU_LABEL, "case:", case, "version:", version)
    run(["nsys", "profile", "--trace=cuda", "--sample=none", "--force-overwrite=true",
         "-o", prefix, "build/stitch_v4", left, right,
         PROFILE_DIR / f"{case}_v{version}.png", "--version", version, "--runs", RUNS],
        log=prefix.with_suffix(".log"))
    report = prefix.with_suffix(".nsys-rep")
    if not report.is_file():
        raise RuntimeError("Thiếu nsys report: " + str(report))
    stats_log = prefix.with_name(prefix.name + "_kernel_stats.log")
    run(["nsys", "stats", "--report", "cuda_gpu_kern_sum", "--format", "csv",
         "--output", "-", report], log=stats_log)
    kernels = read_kernel_summary(stats_log.read_text(encoding="utf-8"))
    for row in kernels:
        row["part"] = blur_part(row["name"])
        row["trace_total_ms"] = row["total_ns"] / 1e6
        row["per_pipeline_ms"] = row["trace_total_ms"] / (RUNS+1)
        row["avg_launch_us"] = row["total_ns"] / row["instances"] / 1e3
    table = pd.DataFrame(kernels).sort_values("total_ns", ascending=False)
    table.to_csv(prefix.with_name(prefix.name + "_kernels.csv"), index=False)
    print("Mọi kernel, xếp theo tổng thời gian trace:")
    display(table[["name", "instances", "avg_launch_us", "per_pipeline_ms", "part"]].round(5))
    parts = {}
    for part in ("coefficients", "horizontal", "vertical"):
        matched = [r for r in kernels if r["part"] == part]
        if sum(r["instances"] for r in matched) != 2*(RUNS+1):
            raise RuntimeError(f"Unexpected launch count for {part}; expected 2*(RUNS+1).")
        parts[part+"_ms"] = sum(r["per_pipeline_ms"] for r in matched)
    if not any("nearest_tiled_kernel" in r["name"] for r in kernels):
        raise RuntimeError("Trace thiếu matching V2; kiểm tra đúng source tích lũy.")
    # Retain memory/API summaries too, including allocation/synchronization.
    extra_log = prefix.with_name(prefix.name + "_api_memory.log")
    extra_code = run(["nsys", "stats", "--report", "cuda_api_sum,cuda_gpu_mem_time_sum",
                      "--format", "csv", "--output", "-", report], check=False, log=extra_log)
    if extra_code:
        print("NOT AVAILABLE: API/memory report; kernel CSV vẫn được lưu.")
    return {"case": case, "version": version, **parts,
            "blur_ms": sum(parts.values()),
            "all_kernel_ms": sum(r["per_pipeline_ms"] for r in kernels)}

profile_rows = []
profile_status = {"requested": RUN_PROFILER, "tool_available": bool(shutil.which("nsys")), "cases": []}
if RUN_PROFILER and shutil.which("nsys"):
    for scene, resolution, case in PROFILE_CASES:
        eligible = summary.loc[summary.case.eq(case) & summary.status.eq("ok") & summary.parity.eq("pass")]
        if eligible.empty:
            profile_status["cases"].append({"case": case, "status": "UNVERIFIED / SKIPPED"})
            continue
        for version in (3,4):
            profile_rows.append(profile_blur(scene, resolution, case, version))
            pd.DataFrame(profile_rows).to_csv(PROFILE_DIR / "blur_parts.csv", index=False)
        profile_status["cases"].append({"case": case, "status": "collected"})
    if profile_rows:
        profiles = pd.DataFrame(profile_rows)
        display(profiles.round(5))
        profile_comparison = []
        for case, group in profiles.groupby("case", sort=False):
            versions = {int(r["version"]): r for r in group.to_dict("records")}
            if not all(v in versions for v in (3,4)):
                continue
            for part in ("coefficients_ms", "horizontal_ms", "vertical_ms", "blur_ms"):
                a,b = versions[3][part], versions[4][part]
                profile_comparison.append({"case": case, "part": part, "V3_ms": a, "V4_ms": b,
                                           "saved_ms": a-b, "speedup": a/b if b>0 else None})
        profile_comparison = pd.DataFrame(profile_comparison)
        profile_comparison.to_csv(PROFILE_DIR / "blur_profile_comparison.csv", index=False)
        display(profile_comparison.round(5))
else:
    print("NOT RUN: Nsight profiling. Cần runtime có nsys; không kết luận nguyên nhân memory/occupancy khi chưa có evidence.")
(PROFILE_DIR / "status.json").write_text(json.dumps(profile_status, indent=2), encoding="utf-8")


## 9. Bàn giao và diễn giải

Tải toàn bộ SESSION_DIR: environment/build flags, command logs (đã giữ stdout/stderr), 36 JSON timing và panorama, summary/timings CSV, inspection CSV và Nsight reports nếu có. Đổi INSPECT_CASE để xem từng case; bật RUN_REVERSE_SWEEP nếu cần kiểm tra bias thứ tự.

Báo cáo riêng: (1) stage và bước chiếm nhiều thời gian ở V3; (2) blur coefficients/horizontal/vertical thay đổi thế nào; (3) blur speedup và E2E speedup; (4) copy/residual và các bước không đổi có biến động không; (5) case chậm hơn và giới hạn kết luận. Blur nhỏ so với E2E thì lợi ích toàn pipeline có thể nhỏ. Không ghép số đo giữa hai GPU session, không tự suy độ lệch chuẩn từ JSON chỉ có trung bình.

Để giữ bằng chứng cũ, số liệu trích từ output notebook đã gửi ngày 24/11/2026 nằm tại reports/v4_saved_run_20261124/metrics_from_notebook.csv; file provenance.json ghi rõ cột bị rút gọn và metadata không được lưu. Số liệu cũ không được dùng thay cho lần chạy mới.
